# Test: seed dimensions

Runs `src/setup`, `src/load_bronze_dims` and `src/load_silver_dims`, then checks that each bronze seed table holds
exactly the rows of its seed CSV and each silver dimension table holds exactly the rows of its seed CSV
(`dim_market`, `dim_commodity`, `dim_date`): `EXCEPT ALL` in both directions returns nothing.
`updated_at` and the `_` metadata columns are not compared; the CSV does not have them.

Tests are plain pytest functions run inside this notebook with `ipytest`.
The notebook fails if any test fails, so it can be used as a job task.

Cleanup: after the tests, the tables and schemas that this run created are dropped.
Existing ones (and their data) are left as they were.


In [ ]:
%pip install pytest ipytest

## State before setup

Recorded before `src/setup` runs, so cleanup only drops what this run creates.

In [ ]:
test_catalog = spark.catalog.currentCatalog()

# Tables and schemas that setup creates, tables first so they are dropped before their schema.
import uuid

RUN_ID = f"pytest-{uuid.uuid4().hex[:8]}"

SETUP_TABLES = [
    f"{test_catalog}.silver.dim_market",
    f"{test_catalog}.silver.dim_commodity",
    f"{test_catalog}.silver.dim_date",
    f"{test_catalog}.bronze.raw_dim_market",
    f"{test_catalog}.bronze.raw_dim_commodity",
    f"{test_catalog}.bronze.raw_dim_date",
    f"{test_catalog}.bronze.raw_mandi_transactions",
]
SETUP_SCHEMAS = [f"{test_catalog}.silver", f"{test_catalog}.bronze"]

tables_existed = {name: spark.catalog.tableExists(name) for name in SETUP_TABLES}
schemas_existed = {
    name: len(spark.sql(f"SHOW SCHEMAS IN {test_catalog} LIKE '{name.split('.')[1]}'").collect()) == 1
    for name in SETUP_SCHEMAS
}

## Run setup and seed

In [ ]:
%run ../src/setup

In [ ]:
dbutils.notebook.run("../src/load_bronze_dims", 600, {"run_id": RUN_ID})
dbutils.notebook.run("../src/load_silver_dims", 600, {"run_id": RUN_ID, "fail_on_error": "true"})

## pytest setup

In [ ]:
import sys

import ipytest
import pytest

# Keep pytest from writing __pycache__ into the Git folder.
sys.dont_write_bytecode = True

ipytest.autoconfig(raise_on_error=True, addopts=["-p", "no:cacheprovider"])

In [ ]:
@pytest.fixture(scope="module", autouse=True)
def drop_created_objects() -> "Generator[None, None, None]":
    """After the tests, drop the tables and schemas, only those this run created.

    No CASCADE: dropping a schema fails instead of removing anything else in it.
    """
    yield

    for name in SETUP_TABLES:
        if tables_existed[name] and "raw_dim_" in name:
            spark.sql(f"DELETE FROM {name} WHERE _run_id = '{RUN_ID}'")
    for name in SETUP_TABLES:
        if not tables_existed[name]:
            spark.sql(f"DROP TABLE IF EXISTS {name}")
    for name in SETUP_SCHEMAS:
        if not schemas_existed[name]:
            spark.sql(f"DROP SCHEMA IF EXISTS {name}")

## Test: tables match file

`SEED_FILES` has the full file path, the bronze table and the silver table of each seed. Both sides are compared as strings:
the CSV as read, the table with its columns cast to STRING (this also checks the casts).

In [ ]:
import os

_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
seed_dir = f"/Workspace{os.path.dirname(os.path.dirname(_notebook_path))}/seed"

# (file path, bronze table, silver table) for each seed.
SEED_FILES = [
    (f"{seed_dir}/dim_market.csv", f"{test_catalog}.bronze.raw_dim_market", f"{test_catalog}.silver.dim_market"),
    (f"{seed_dir}/dim_commodity.csv", f"{test_catalog}.bronze.raw_dim_commodity", f"{test_catalog}.silver.dim_commodity"),
    (f"{seed_dir}/dim_date.csv", f"{test_catalog}.bronze.raw_dim_date", f"{test_catalog}.silver.dim_date"),
]


def read__seed_file(file_path: str) -> "DataFrame":
    """Read a seed CSV with all columns as STRING."""
    return (
        spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"file:{file_path}" if file_path.startswith("/Workspace") else file_path)
    )


def as_strings(table_name: str, drop: list[str]) -> "DataFrame":
    """Read a table with the given columns dropped and the rest cast to STRING."""
    df = spark.table(table_name).drop(*drop)
    return df.select(*[df[c].cast("string").alias(c) for c in df.columns])


@pytest.mark.parametrize("file_path, bronze_table, silver_table", SEED_FILES)
def test__tables_match_seed_file(file_path: str, bronze_table: str, silver_table: str) -> None:
    """Bronze (this run) and silver have exactly the rows of the seed file (EXCEPT ALL, both ways)."""
    file_df = read__seed_file(file_path)
    bronze_df = (
        spark.table(bronze_table)
        .where(f"_run_id = '{RUN_ID}'")
        .drop("_ingested_at", "_source_mode", "_file_name", "_run_id", "_row_no")
    )
    silver_df = as_strings(silver_table, ["updated_at"])

    assert file_df.count() > 0
    assert file_df.exceptAll(bronze_df).count() == 0
    assert bronze_df.exceptAll(file_df).count() == 0
    assert file_df.exceptAll(silver_df).count() == 0
    assert silver_df.exceptAll(file_df).count() == 0

## Run

In [ ]:
ipytest.run("-v")